# Liver Disease Prediction Model Training

This notebook outlines the process of training an XGBoost classifier to predict liver disease. The workflow includes:
1. Setting up the environment and importing necessary libraries.
2. Loading data from a CSV file (`syn_liver_disease_data.csv`).
3. Preprocessing the data, which involves separating features and target, splitting into training and testing sets, and feature scaling.
4. Training an XGBoost model with hyperparameter tuning using 5-fold cross-validation and GridSearchCV.
5. Evaluating the trained model on the test set using various metrics.
6. Analyzing feature importances from the model.
7. Saving the trained scaler and model for future use.

## 0. Setup and Imports

This cell imports all necessary libraries and sets up the environment. 
- `%matplotlib inline` ensures plots are displayed directly within the notebook.
- Warnings are configured to suppress common `UserWarning` from XGBoost and general `FutureWarning` for cleaner output.

In [ ]:
%matplotlib inline 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

import xgboost as xgb
import joblib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost') 
warnings.filterwarnings('ignore', category=FutureWarning) 

print("All libraries imported and warnings configured.")

## 1. Data Loading and Initial Preprocessing

This section handles loading the dataset from a CSV file. 
- A `DATA_PATH` variable is defined for easy modification of the data source.
- The dataset is loaded into a pandas DataFrame. Basic information like shape and head of the data is printed.
- Features (`X`) and the target variable (`y`) are separated. 
- The data is split into training and testing sets *before* any scaling is applied to prevent data leakage. `stratify=y` is used to ensure proportional class representation in splits.
- `StandardScaler` is initialized and fit *only* on the training data (`X_train`), then used to transform both `X_train` and `X_test`.
- Variables are initialized to `None` to ensure they exist even if parts of the cell fail, aiding robust error checking in subsequent cells.

In [ ]:
DATA_PATH = 'syn_liver_disease_data.csv'
data = None 
X, y = None, None 
X_train, X_test, y_train, y_test = None, None, None, None
X_train_scaled, X_test_scaled = None, None
scaler = None

try:
    data = pd.read_csv(DATA_PATH)
    print(f"Successfully loaded data from {DATA_PATH}")
    print(f"Dataset shape: {data.shape}")
    print("First 5 rows of the dataset:")
    print(data.head())
except FileNotFoundError:
    print(f"ERROR: The file {DATA_PATH} was not found. Please ensure it's in the correct directory.")
except Exception as e:
    print(f"ERROR: An error occurred while loading the data: {e}")

if data is not None:
    try:
        if 'Diagnosis' in data.columns:
            X = data.drop(columns=['Diagnosis'])
            y = data['Diagnosis']
            print("\nFeatures (X) and target (y) defined.")
            print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")
        else:
            print("ERROR: 'Diagnosis' column not found in the dataset.")
            X, y = None, None # Ensure X, y are None if 'Diagnosis' is missing
    except Exception as e:
        print(f"ERROR: An error occurred while defining X and y: {e}")
        X, y = None, None

if X is not None and y is not None:
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print("\nData split into training and testing sets.")
        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("\nData scaled using StandardScaler (fit on train, transformed train and test).")
        print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
        print(f"Shape of X_test_scaled: {X_test_scaled.shape}")
    except Exception as e:
        print(f"ERROR: An error occurred during train/test split or scaling: {e}")
else:
    print("\nSkipping train/test split and scaling due to previous errors in data loading or X/y definition.")

## 2. Model Training with Hyperparameter Tuning (GridSearchCV)

This section focuses on training the XGBoost classifier. 
- A parameter grid (`param_grid`) is defined for `n_estimators`, `max_depth`, and `learning_rate`.
- `xgb.XGBClassifier` is initialized with `use_label_encoder=False`, `eval_metric='logloss'`, and a `random_state` for reproducibility.
- 5-fold stratified cross-validation (`StratifiedKFold`) is set up to ensure that class proportions are maintained in each fold, which is important for potentially imbalanced datasets.
- `GridSearchCV` is used to systematically search for the best hyperparameter combination based on `roc_auc` scoring. `n_jobs=-1` uses all available CPU cores.
- The best estimator from `GridSearchCV` is then selected. 
- Finally, the scaler (fit on `X_train`) and the best XGBoost model are saved to disk using `joblib` for persistence.

In [ ]:
best_xgb_model = None
grid_search = None

if X_train_scaled is not None and y_train is not None and scaler is not None:
    try:
        param_grid = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2]
        }

        xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        grid_search = GridSearchCV(estimator=xgb_clf, param_grid=param_grid, cv=cv, scoring='roc_auc', verbose=1, n_jobs=-1)
        
        print("Starting GridSearchCV for XGBoost...")
        grid_search.fit(X_train_scaled, y_train)

        best_xgb_model = grid_search.best_estimator_
        print("\nGridSearchCV complete.")
        print("Best Hyperparameters found:", grid_search.best_params_)

        # Save the scaler and the best model
        joblib.dump(scaler, 'scaler.pkl')
        print("Scaler saved to scaler.pkl")
        joblib.dump(best_xgb_model, 'xgb_model.pkl')
        print("Best XGBoost model saved to xgb_model.pkl")

    except Exception as e:
        print(f"ERROR: An error occurred during model training or saving: {e}")
else:
    print("Skipping model training and saving due to errors in data loading/preprocessing.")

## 3. Model Evaluation

The trained model (with the best hyperparameters from GridSearchCV) is evaluated on the unseen test set (`X_test_scaled`, `y_test`). 
Metrics calculated include:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC Score (requires probability predictions for the positive class)
- A confusion matrix is also plotted for a visual representation of classification performance (True Positives, False Positives, True Negatives, False Negatives).

In [ ]:
if best_xgb_model is not None and X_test_scaled is not None and y_test is not None:
    try:
        print("\nEvaluating model on the test set...")
        y_pred = best_xgb_model.predict(X_test_scaled)
        y_pred_proba = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

        print('\nModel Evaluation Metrics:')
        print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
        print(f"  Recall: {recall_score(y_test, y_pred):.4f}")
        print(f"  F1 Score: {f1_score(y_test, y_pred):.4f}")
        print(f"  ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

        # Plotting Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'])
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()
    except Exception as e:
        print(f"ERROR: An error occurred during model evaluation: {e}")
else:
    print("Skipping model evaluation due to errors in previous steps (model not trained or test data unavailable).")

## 4. Feature Importance

Understanding which features are most influential in the model's predictions is crucial for model interpretation and potentially for feature selection in future iterations.
- Feature importances are extracted from the `best_xgb_model`.
- These importances are then paired with their corresponding feature names (obtained from the columns of the original `X` DataFrame).
- The features are displayed in a sorted list and visualized using a horizontal bar plot for easier interpretation.

In [ ]:
if best_xgb_model is not None and X is not None: # X is needed for feature names
    try:
        print("\nCalculating and displaying feature importances...")
        importances = best_xgb_model.feature_importances_
        feature_names = X.columns

        feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

        print("\nFeature Importances (most important first):")
        print(feature_importance_df)

        plt.figure(figsize=(12, 8))
        sns.barplot(x='importance', y='feature', data=feature_importance_df, palette='viridis')
        plt.title('Feature Importance from XGBoost Model')
        plt.xlabel('Importance Score')
        plt.ylabel('Features')
        plt.tight_layout() # Adjust layout to prevent labels from overlapping
        plt.show()
    except Exception as e:
        print(f"ERROR: An error occurred during feature importance calculation: {e}")
else:
    print("Skipping feature importance calculation due to errors in previous steps (model not trained or X unavailable).")

## 5. Conclusion

This notebook demonstrated the end-to-end process of training an XGBoost model for liver disease prediction. Key steps included:
- Loading and robustly preprocessing the data.
- Performing hyperparameter tuning using `GridSearchCV` with 5-fold stratified cross-validation.
- Evaluating the model's performance on a held-out test set.
- Visualizing feature importances to understand model drivers.
- Saving the trained `StandardScaler` and `XGBClassifier` model for deployment or further analysis.

The results from the evaluation metrics and feature importances provide insights into the model's predictive capabilities and the relative significance of different input features.